<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">


# Python for Finance, 3rd Edition
## Chapter 08 · Data Visualization
&copy; Dr. Yves J. Hilpisch<br>
AI-supported by GPT 5.x<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
This notebook mirrors the chapter examples in a Colab-ready format so that you
can run, tweak, and extend them interactively.


### How to Use This Notebook
- Run the cells top to bottom the first time so the plots build in the right
  order.
- Add your own cells for experiments or refactorings.
- Use the book text for the surrounding discussion and design choices.


In [ ]:
from pathlib import Path
import subprocess
import sys

NOTEBOOK_SUBDIR = "notebooks"
COLAB_PACKAGES = {}
REPO_NAME = "py4fi3rd"
REPO_URL = "https://github.com/yhilpisch/py4fi3rd.git"


def _support_dir() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        support_dir = candidate / "notebooks"
        if (support_dir / "_book_notebook_support.py").exists():
            return support_dir
    if "google.colab" in sys.modules:
        root = Path("/content") / REPO_NAME
        if not root.exists():
            subprocess.run(
                ["git", "clone", "--depth", "1", REPO_URL, str(root)],
                check=True,
            )
        return root / "notebooks"
    raise RuntimeError("Could not locate notebook support helpers.")


SUPPORT_DIR = _support_dir()
if str(SUPPORT_DIR) not in sys.path:
    sys.path.insert(0, str(SUPPORT_DIR))

from _book_notebook_support import setup_notebook

CONTEXT = setup_notebook(
    notebook_subdir=NOTEBOOK_SUBDIR,
    colab_packages=COLAB_PACKAGES,
)

PROJECT_ROOT = CONTEXT["PROJECT_ROOT"]
NOTEBOOK_DIR = CONTEXT["NOTEBOOK_DIR"]
CODE_DIR = CONTEXT["CODE_DIR"]
CHAPTERS_DIR = CONTEXT["CHAPTERS_DIR"]
FIGURES_DIR = CONTEXT["FIGURES_DIR"]
DATA_DIR = CONTEXT["DATA_DIR"]

PROJECT_ROOT

This chapter shows how `matplotlib` turns numeric series into figures that are
easier to inspect and share.


# Why Visualization Matters in Finance


Good plots make trends, outliers, and regime changes easier to spot than
tables alone.


# Matplotlib Basics


Start with the object-oriented figure and axes workflow for clear, reusable
plots.


## A First Line Plot


Build a simple synthetic price series and plot it with a shared style.


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
mpl.style.use("seaborn-v0_8")
mpl.rcParams.update({"font.family": "serif", "figure.dpi": 300})

In [ ]:
rng = np.random.default_rng(seed=42)
n = 100
# Draw small, normally distributed percentage changes as a simple return series.
steps = rng.normal(loc=0.0, scale=0.01, size=n)
# Build a synthetic price path by compounding the returns from a starting
# price of 100.
prices = 100 * (1 + steps).cumprod()
fig, ax = plt.subplots()
# Plot the price series as a blue line with a slightly thicker linewidth.
ax.plot(prices, color="tab:blue", linewidth=1.25)
ax.set_title("Synthetic Price Series")
ax.set_xlabel("Time")
ax.set_ylabel("Price")
ax.grid(True, linestyle="--", alpha=0.3)

# Plotting Time Series from pandas


`pandas` series know how to plot themselves while still giving you access to
the axes.


## Line Plots from a Series


Use a date index so the horizontal axis shows time naturally.


In [ ]:
import pandas as pd

In [ ]:
# Create a short business-day `DatetimeIndex`.
dates = pd.date_range("2026-01-01", periods=5, freq="B")
prices = pd.Series(
    [100.0, 101.5, 99.0, 102.0, 103.5],
    index=dates,
    name="price",
)
ax = prices.plot(figsize=(6, 3))
ax.set_title("Daily Prices")
ax.set_ylabel("Price")


# Histograms and Distributions


Histograms summarize the shape of return distributions at a glance.


## A Simple Returns Histogram


Compute returns first, then inspect their spread with a histogram.


In [ ]:
rets = prices.pct_change().dropna()
fig, ax = plt.subplots(figsize=(6, 3))
# Plot a histogram with 10 bins, gray bars, and black edges for clarity.
ax.hist(rets, bins=10, color="tab:gray", edgecolor="black", alpha=0.7)
ax.set_title("Daily Returns Histogram")
ax.set_xlabel("Return")
ax.set_ylabel("Frequency")

# Multi-Panel Figures


Multiple panels help you compare related views without mixing scales.


## Prices and Returns in One Figure


Put price levels and returns in one figure when they tell a single story.


In [ ]:
# Create a figure with two stacked axes that share the same x-axis.
fig, (ax_price, ax_ret) = plt.subplots(2, 1, figsize=(6, 4), sharex=True)
ax_price.plot(prices, color="tab:blue")
ax_price.set_title("Prices and Returns")
ax_price.set_ylabel("Price")
# Plot returns as a bar chart in the second axis for better visual separation.
ax_ret.bar(rets.index, rets, color="tab:orange", width=0.8)
# Draw a horizontal zero line to distinguish positive and negative returns.
ax_ret.axhline(0.0, color="black", linewidth=0.8)
ax_ret.set_ylabel("Return")
fig.tight_layout()

## Line and Bar Subplots


Combine line and bar views to compare two series with a shared x-axis.


In [ ]:
rng = np.random.default_rng(seed=21)
n = 50
x = np.arange(n)
# Build a cumulative series that is suitable for a line plot.
y_line = np.cumsum(rng.normal(loc=0.0, scale=1.0, size=n))
# Create a sequence of independent values that works well as bar heights.
y_bar = rng.normal(loc=0.0, scale=1.0, size=n)
# Create two vertically stacked subplots that share the same x-axis.
fig, (ax_line, ax_bar) = plt.subplots(2, 1, figsize=(6, 4), sharex=True)
# Plot the cumulative series as a line with markers in the top subplot.
ax_line.plot(x, y_line, color="tab:blue", marker="o", markersize=3)
ax_line.set_ylabel("Line value")
ax_line.set_title("Line and Bar Subplots")
# Plot individual values as bars in the bottom subplot.
ax_bar.bar(x, y_bar, color="tab:green", width=0.9)
ax_bar.set_ylabel("Bar value")
ax_bar.set_xlabel("Index")
fig.tight_layout()

# Different Scales and Dual Axes


Dual axes help when two series move on very different scales.


In [ ]:
rng = np.random.default_rng(seed=7)
n = 100
x = np.arange(n)
# Generate a series with relatively large magnitude.
y1 = np.cumsum(rng.normal(loc=0.0, scale=1.0, size=n))
# Generate a second series that is scaled down by a factor of 100.
y2 = 0.01 * np.cumsum(rng.normal(loc=0.0, scale=1.0, size=n))
fig, ax1 = plt.subplots(figsize=(6, 3))
ax2 = ax1.twinx()
# Plot the large-scale series against the left y-axis.
ax1.plot(x, y1, color="tab:blue", label="Series A (large scale)")
ax1.set_xlabel("Time")
ax1.set_ylabel("Series A", color="tab:blue")
# Plot the small-scale series against the right y-axis.
ax2.plot(
    x,
    y2,
    color="tab:orange",
    linestyle="--",
    label="Series B (small scale)",
)
ax2.set_ylabel("Series B", color="tab:orange")
# Keep both legends visible on the primary axes.
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")
ax1.grid(True, linestyle="--", alpha=0.3)
fig.tight_layout()


# Scatter Plots for Relationships


Scatter plots expose relationships between two variables without forcing a
time axis.


In [ ]:
rng = np.random.default_rng(seed=42)
x = rng.normal(loc=0.0, scale=0.01, size=500)
# Draw independent noise with smaller variability than the signal.
noise = rng.normal(loc=0.0, scale=0.002, size=500)
# Construct `y` as a linear function of `x` plus noise, so that the
# variables are positively correlated but not perfectly aligned.
y = 0.5 * x + noise
fig, ax = plt.subplots(figsize=(4.5, 4.5))
# Plot the signal pairs as a scatter cloud with small, semi-transparent markers.
ax.scatter(x, y, s=10, alpha=0.6, color="tab:blue", edgecolors="none")
# Add a title and axis labels to clarify the relationship being shown.
ax.set_title("Linear Relationship with Noise")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.axhline(0.0, color="black", linewidth=0.8)
ax.axvline(0.0, color="black", linewidth=0.8)
ax.grid(True, linestyle="--", alpha=0.3)

# Three-Dimensional Views


Three-dimensional plots are useful when a surface or sampled cloud really
needs three axes.


## A 3D Surface Plot


A smooth surface can stand in for a stylized pricing or volatility surface.


In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
x = np.linspace(-2.0, 2.0, 60)
y = np.linspace(-2.0, 2.0, 60)
# Create a grid of x/y coordinates that covers the domain of interest.
X, Y = np.meshgrid(x, y)
# Evaluate a smooth function on the grid to obtain height values.
Z = np.exp(-0.5 * (X**2 + Y**2))
fig = plt.figure(figsize=(6, 4))
ax = fig.add_subplot(111, projection="3d")
# Plot the surface with a colormap and slight transparency.
surf = ax.plot_surface(X, Y, Z, cmap="viridis", edgecolor="none", alpha=0.9)
ax.set_title("3D Gaussian Surface")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
fig.colorbar(surf, ax=ax, shrink=0.7, aspect=10)
fig.tight_layout()

## A 3D Scatter Plot


Sampled 3D points can show shape without implying a smooth surface between
observations.


In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
rng = np.random.default_rng(seed=123)
n = 400
# Sample maturities from a range that might correspond to short and long tenors.
x = rng.uniform(0.5, 2.0, size=n)
# Sample strikes from a range around the at-the-money level.
y = rng.uniform(0.1, 1.0, size=n)
# Compute synthetic values that decay with both maturity and strike plus a
# bit of noise.
z = np.exp(-0.5 * (x + y)) + 0.05 * rng.standard_normal(size=n)
fig = plt.figure(figsize=(6, 4))
ax = fig.add_subplot(111, projection="3d")
# Color-code the points by their `z` value to reveal patterns in all three
# dimensions.
sc = ax.scatter(x, y, z, c=z, cmap="viridis", s=15, alpha=0.8)
ax.set_title("3D Scatter Plot")
ax.set_xlabel("Maturity")
ax.set_ylabel("Strike")
ax.set_zlabel("Value")
fig.colorbar(sc, ax=ax, shrink=0.7, aspect=10)
fig.tight_layout()

# Where We Are Heading Next


The plotting patterns here reappear later when we visualize time
series, strategy results, and risk measures.


## Figure Generation (Optional)
Run the chapter's figure scripts under `code/figures/` to regenerate the PNG
files under `assets/figures/`.


In [ ]:
import runpy

scripts = [
    "../code/figures/ch08_dual_axes.py",
    "../code/figures/ch08_line_and_bar_subplots.py",
    "../code/figures/ch08_prices_and_returns.py",
    "../code/figures/ch08_prices_line.py",
    "../code/figures/ch08_returns_hist.py",
    "../code/figures/ch08_scatter_3d.py",
    "../code/figures/ch08_scatter_returns.py",
    "../code/figures/ch08_styles_and_legend.py",
    "../code/figures/ch08_surface_3d.py",
]

for script in scripts:
    try:
        runpy.run_path(script, run_name="__main__")
        print(f"OK: {script}")
    except ModuleNotFoundError as e:
        print(f"Skipping {script}: missing dependency ({e.name}).")
    except Exception as e:
        print(f"Failed {script}: {type(e).__name__}: {e}")


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">
